# Purge Archived Cycles

Selectively delete archived cycles from your Risk Modeling workflow.

1. Displays all archived cycles with their details
2. Prompts for confirmation before deleting each cycle
3. Removes both the database records and the archived directory for confirmed deletions

In [ ]:
import shutil
from helpers import cycle, constants, ux

In [ ]:
all_cycles_df = cycle.get_cycle_status()
archived_df = all_cycles_df[all_cycles_df['status'] == 'ARCHIVED']

if archived_df.empty:
    ux.info("No archived cycles found. Nothing to purge.")
else:
    ux.warning(f"Found {len(archived_df)} archived cycle(s)")
    ux.dataframe(archived_df, title="Archived Cycles")

In [ ]:
deleted_count = 0
skipped_count = 0

if not archived_df.empty:
    for _, row in archived_df.iterrows():
        cycle_name = row['cycle_name']
        cycle_id = row['id']

        ux.subheader(f"Cycle: {cycle_name}")
        ux.display_key_value({
            "Created": ux.format_timestamp(row['created_ts']),
            "Archived": ux.format_timestamp(row['archived_ts'])
        })

        if ux.yes_no(f"Delete '{cycle_name}'?"):
            # Remove archived directory if it exists
            archive_dir = constants.ARCHIVE_PATH / cycle_name
            if archive_dir.exists():
                shutil.rmtree(archive_dir)

            # Remove database records
            cycle.delete_cycle(cycle_id)
            ux.success(f"Deleted: {cycle_name}")
            deleted_count += 1
        else:
            ux.info(f"Skipped: {cycle_name}")
            skipped_count += 1

    print()
    ux.header("Purge Complete")
    ux.success(f"Deleted {deleted_count} cycle(s), skipped {skipped_count} cycle(s)")